In [0]:
# ---------------------------------------------------------------------------
# SILVER: flatten the hourly arrays in the Open-Meteo response into one row
# per (city, hour), cast types, dedupe, and upsert into a clean Delta table.
# ---------------------------------------------------------------------------

dbutils.widgets.text("bronze_table", "bronze.bronze_table", "Bronze Delta table")
dbutils.widgets.text("silver_table", "silver.weather_hourly", "Silver Delta table")

bronze_table = dbutils.widgets.get("bronze_table")
silver_table = dbutils.widgets.get("silver_table")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

df_bronze = spark.table(bronze_table)

# Open-Meteo returns parallel arrays under "hourly": time[], temperature_2m[], etc.
# arrays_zip pairs them up positionally so we can explode into rows.
df_exploded = (
    df_bronze
    .select(
        "city",
        F.arrays_zip(
            "hourly.time",
            "hourly.temperature_2m",
        ).alias("hourly_zip"),
    )
    .withColumn("hourly_row", F.explode("hourly_zip"))
    .select(
        "city",
        F.to_timestamp(F.col("hourly_row.time")).alias("observation_ts"),
        F.col("hourly_row.temperature_2m").cast("double").alias("temperature_c"),
    )
)

df_silver = (
    df_exploded
    .dropDuplicates(["city", "observation_ts"])
    .withColumn("_processed_ts", F.current_timestamp())
)

display(df_silver)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

if spark.catalog.tableExists(silver_table):
    target = DeltaTable.forName(spark, silver_table)
    (
        target.alias("t")
        .merge(df_silver.alias("s"), "t.city = s.city AND t.observation_ts = s.observation_ts")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_silver.write.format("delta").saveAsTable(silver_table)

row_count = df_silver.count()
print(f"Silver upsert complete: {row_count} hourly records merged into {silver_table}")

In [0]:
%sql
select * from devdatabricksweather.silver.weather_hourly